# Домашнее задание 2. Отбор кандидатов (20p)

В этом задании мы построим первый этап рекомендательной системы — retrieval, то есть отбор небольшого подмножества кандидатов из полного каталога. Следующий этап, ranking, сможет изменить их порядок, но не сможет вернуть товар, который в список кандидатов не попал. Поэтому качество retrieval задаёт верхнюю границу качества всей системы.



Пусть каждая модель возвращает не более 50 кандидатов, а все метрики вычисляются на срезах `@20` и `@50`. Для каждого семейства моделей будем использовать один и тот же пайплайн:

1. реализовать алгоритм и проверить отдельные шаги на небольших данных;
2. обучить все заданные конфигурации на `train` и оценить их на `validation`;
3. выбрать конфигурацию с максимальным `Recall@20`;
4. один раз обучить модель на `train + validation` и оценить на `test`;
5. сохранить результат в словаре `METRICS` для итогового сравнения семейств моделей.

Выборка `test` не используется ни для выбора модели, ни для настройки
гиперпараметров!

In [ ]:
import math
import random
import time
from abc import ABC, abstractmethod
from collections.abc import Callable, Iterable, Mapping, Sequence
from pathlib import Path
from typing import Any, Literal, Self, cast

import implicit
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import tests
import torch
import torch.nn.functional as F
from numpy.typing import NDArray
from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path("../data")
interactions = pl.read_parquet(DATA_DIR / "interactions.parquet")
items = pl.read_parquet(DATA_DIR / "items.parquet")
users = pl.read_parquet(DATA_DIR / "users.parquet")

interactions = interactions.with_columns(
    pl.col("user_id").cast(pl.Utf8),
    pl.col("item_id").cast(pl.Utf8),
    pl.col("rating").cast(pl.Float64),
    pl.col("timestamp").cast(pl.Datetime("us", "UTC")),
)

users = users.with_columns(pl.col("user_id").cast(pl.Utf8)).filter(
    pl.col("user_id").is_in(interactions["user_id"].unique().to_list())
)
items = items.with_columns(pl.col("item_id").cast(pl.Utf8)).filter(
    pl.col("item_id").is_in(interactions["item_id"].unique().to_list())
)

## 0. Разминка: от циклов к матричным операциям (1p)

В рекомендациях почти любая формула «для каждого пользователя посчитать что-то по
всем товарам» переписывается в одно матричное выражение — и работает на порядки
быстрее. Потренируемся на игрушечных
данных.


### 0.1. Популярность товаров

Пусть `matrix` — бинарная матрица взаимодействий `(число пользователей, число
товаров)`. Популярность товара — это число пользователей, которые с ним
взаимодействовали, то есть сумма по столбцу.

Нужно реализовать две версии одной и той же функции: `popularity_by_loop` — явным двойным циклом по
пользователям и товарам, `popularity_vectorized` — одним выражением над `matrix`.


In [ ]:
def popularity_by_loop(matrix: np.ndarray) -> np.ndarray:
    """Сумма событий по товарам, посчитанная явным двойным циклом."""
    # PUT YOUR CODE HERE
    return


def popularity_vectorized(matrix: np.ndarray) -> np.ndarray:
    """То же самое одним матричным выражением."""
    # PUT YOUR CODE HERE
    return

In [ ]:
bench_matrix = np.random.default_rng(0).integers(
    0, 2, size=(200, 2_000)
).astype(np.float32)

started = time.perf_counter()
loop_popularity = popularity_by_loop(bench_matrix)
loop_seconds = time.perf_counter() - started

started = time.perf_counter()
vectorized_popularity = popularity_vectorized(bench_matrix)
vectorized_seconds = time.perf_counter() - started

assert np.allclose(loop_popularity, vectorized_popularity, atol=1e-3)
print(f"Cycle: {loop_seconds:.3f}s, vectorized: {vectorized_seconds:.4f}s")


### 0.2. Оценки «пользователь × товар»

Пусть `user_factors` имеет форму `(число пользователей, d)`, а `item_factors` —
`(число товаров, d)`. Оценка релевантности пары — скалярное произведение их
векторов, $s_{ui}=x_u^Ty_i$.

Посчитаем оценки сразу для всех пар. Нужно реализовать `user_item_scores_by_loop` двойным циклом и
`user_item_scores_vectorized` одним матричным умножением.

In [ ]:
def user_item_scores_by_loop(
    user_factors: np.ndarray, item_factors: np.ndarray
) -> np.ndarray:
    """s_ui = x_u^T y_i, посчитанное двойным циклом по пользователям и товарам."""
    # PUT YOUR CODE HERE
    return


def user_item_scores_vectorized(
    user_factors: np.ndarray, item_factors: np.ndarray
) -> np.ndarray:
    """То же самое одним матричным умножением."""
    # PUT YOUR CODE HERE
    return

In [ ]:
bench_users = np.random.default_rng(0).normal(size=(200, 32)).astype(np.float32)
bench_items = np.random.default_rng(1).normal(size=(2_000, 32)).astype(np.float32)

started = time.perf_counter()
loop_scores = user_item_scores_by_loop(bench_users, bench_items)
loop_seconds = time.perf_counter() - started

started = time.perf_counter()
vectorized_scores = user_item_scores_vectorized(bench_users, bench_items)
vectorized_seconds = time.perf_counter() - started

assert np.allclose(loop_scores, vectorized_scores, atol=1e-3)
print(f"Cycle: {loop_seconds:.3f}s, vectorized: {vectorized_seconds:.4f}s")

## 1. Разведочный анализ данных (2p)

Перед обучением моделей полезно провести разведочный анализ. Он помогает понять структуру данных и сформулировать первые гипотезы.

Обратите внимание, что все операции необходимо выполнять на `polars`!


### 1.1. Распределение оценок

Обозначим множество взаимодействий через $E$, а его размер — через $N=|E|$. Для
каждой оценки $r\in\{1,2,3,4,5\}$ вычислим число событий $n_r$ и их долю

$$p_r=\frac{n_r}{N}.$$

Здесь $n_r$ — число взаимодействий с оценкой $r$, а $p_r$ — доля таких
взаимодействий среди всех событий.

In [ ]:
def rating_distribution(frame: pl.DataFrame) -> pl.DataFrame:
    # PUT YOUR CODE HERE
    return


tests.check_rating_distribution(rating_distribution)

### 1.2. Активность пользователей

Пусть $E$ — множество взаимодействий, а $u_e$ — пользователь в событии $e$.
**Активность пользователя** $u$ равна

$$a_u=|\{e\in E:u_e=u\}|,$$

то есть числу его взаимодействий.

Вычислим $a_u$ и построим таблицу частот: для каждого значения активности
укажем число и долю пользователей с таким числом событий.


In [ ]:
def user_activity_distribution(frame: pl.DataFrame) -> pl.DataFrame:
    # PUT YOUR CODE HERE
    return


tests.check_user_activity_distribution(user_activity_distribution)

### 1.3. Популярность товаров и длинный хвост

Пусть $E$ — множество взаимодействий, а $i_e$ — товар в событии $e$. **Популярность** товара $i$ равна

$$q_i=|\{e\in E:i_e=i\}|,$$

то есть числу взаимодействий с ним.

Товары сортируются по убыванию $q_i$. Если $rank_i$ — позиция товара в этом порядке, а $|I|$ —
число товаров, то **процентильный ранг**

$$z_i=\frac{rank_i-1}{|I|-1}$$

переводит позицию в диапазон $[0,1]$. **Накопленная доля**

$$c_i=\frac{\sum_{j:rank_j\leq rank_i}q_j}{|E|}$$

показывает, какая часть всех событий приходится на товары не менее популярные, чем
$i$.

**Головой** каталога называется минимальный префикс, для которого $c_i\geq0.8$;
остальные товары образуют **длинный хвост**.

Построим таблицу: для каждого `item_id` рассчитаем его популярность, процентильный ранг и накопленную долю событий. Товары с одинаковым числом взаимодействий сортируются по возрастанию `item_id`.

In [ ]:
def item_popularity_curve(frame: pl.DataFrame) -> pl.DataFrame:
    # PUT YOUR CODE HERE
    return


tests.check_item_popularity_curve(item_popularity_curve)

### 1.4. Динамика событий

Для календарного месяца $m$ вычислим $n_m$ — число событий, временная метка
`timestamp` которых попадает в этот месяц. Так видны тренд, сезонность и пропуски в сборе данных.


In [ ]:
def events_by_month(frame: pl.DataFrame) -> pl.DataFrame:
    # PUT YOUR CODE HERE
    return

tests.check_events_by_month(events_by_month)

### 1.5. Визуализация

Соберём всё вместе.


In [ ]:
def make_eda_figure(frame: pl.DataFrame) -> tuple[plt.Figure, dict[str, pl.DataFrame]]:
    tables = {
        "ratings": rating_distribution(frame),
        "activity": user_activity_distribution(frame),
        "popularity": item_popularity_curve(frame),
        "monthly": events_by_month(frame),
    }
    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    rating_axis, activity_axis, popularity_axis, month_axis = axes.ravel()

    rating_axis.bar(
        tables["ratings"]["rating"].to_numpy(), tables["ratings"]["n_events"].to_numpy()
    )
    rating_axis.set(
        title="Распределение оценок", xlabel="Оценка", ylabel="Число событий"
    )

    activity_axis.plot(
        tables["activity"]["activity"].to_numpy(),
        tables["activity"]["n_users"].to_numpy(),
    )
    activity_axis.set(
        title="Активность пользователей",
        xlabel="Число событий пользователя",
        ylabel="Число пользователей",
        xscale="log",
        yscale="log",
    )

    popularity_axis.plot(
        100 * tables["popularity"]["percentile_rank"].to_numpy(),
        tables["popularity"]["n_interactions"].to_numpy(),
    )
    popularity_axis.set(
        title="Популярность товаров",
        xlabel="Процентильный ранг, %",
        ylabel="Число взаимодействий",
        yscale="log",
    )

    month_axis.plot(
        tables["monthly"]["month"].to_numpy(), tables["monthly"]["n_events"].to_numpy()
    )
    month_axis.set(title="События во времени", xlabel="Месяц", ylabel="Число событий")
    month_axis.tick_params(axis="x", rotation=45)
    figure.tight_layout()
    return figure, tables


eda_figure, eda_tables = make_eda_figure(interactions)
plt.show()

In [ ]:
popularity = eda_tables["popularity"]
head_end = int(
    np.searchsorted(popularity["cumulative_event_share"].to_numpy(), 0.8) + 1
)
eda_summary = pd.Series(
    {
        "n_events": len(interactions),
        "n_users": interactions["user_id"].n_unique(),
        "n_items": interactions["item_id"].n_unique(),
        "median_user_activity": interactions.group_by("user_id")
        .agg(pl.len())
        .select(pl.col("len").median())
        .item(),
        "median_item_popularity": interactions.group_by("item_id")
        .agg(pl.len())
        .select(pl.col("len").median())
        .item(),
        "head_items": head_end,
        "head_catalog_share": head_end / len(popularity),
        "tail_items": len(popularity) - head_end,
        "tail_catalog_share": 1 - head_end / len(popularity),
    },
    name="value",
)
eda_summary

## 2. Глобальное временное разбиение (1p)

Офлайн-оценка должна воспроизводить реальный порядок событий: модель знает прошлое и
предсказывает будущее. Поэтому все события сортируются глобально по времени, а не
разбиваются отдельно для каждого пользователя.


### 2.1. Разбиение на `train`, `validation` и `test`

Функция принимает три доли: $r_{train}$ для обучающей выборки, $r_{val}$ для
валидационной и $r_{test}$ для тестовой. Каждая доля должна лежать строго между 0 и
1, а их сумма должна удовлетворять условию

$$r_{train}+r_{val}+r_{test}=1.$$

После глобальной сортировки по времени первые $\lfloor Nr_{train}\rfloor$ строк
образуют `train`, следующие $\lfloor Nr_{val}\rfloor$ — `validation`, а остальные —
`test`; здесь $N$ — общее число событий. Остаток передаётся в `test`, чтобы ни одна
строка не потерялась из-за округления.

Если временные метки `timestamp` совпадают, исходный индекс строки задаёт детерминированный дополнительный порядок.

In [ ]:
def temporal_ratio_split(
    frame: pl.DataFrame,
    train_ratio: float = 0.8,
    validation_ratio: float = 0.1,
    test_ratio: float = 0.1,
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    ratios = np.asarray([train_ratio, validation_ratio, test_ratio], dtype=float)
    if not np.all((ratios > 0) & (ratios < 1)):
        raise ValueError("Каждая доля должна лежать строго между 0 и 1")
    if not np.isclose(ratios.sum(), 1.0, rtol=0.0, atol=1e-10):
        raise ValueError("Сумма долей train, validation и test должна быть равна 1")
    # PUT YOUR CODE HERE
    return


tests.check_temporal_ratio_split(temporal_ratio_split)

Применим к нашим данным. Обратите внимание, что факт оценки считается за положительное событие, а числовое значение оценки - за степень выраженности положительного сигнала.

In [ ]:
train, validation, test = temporal_ratio_split(
    interactions,
    train_ratio=0.8,
    validation_ratio=0.1,
    test_ratio=0.1,
)
train_validation = pl.concat([train, validation])

print(len(train), len(validation), len(test))

### 2.2. Диагностика холодных пользователей и товаров

Пусть $U_{train}$ и $U_{eval}$ — множества пользователей в обучающем и оцениваемом
периодах. Доля холодных (новых) пользователей равна

$$\frac{|U_{eval}\setminus U_{train}|}{|U_{eval}|}.$$

Для товаров используется та же формула с множествами $I_{train}$ и $I_{eval}$.

Вычислим доли холодных пользователей и товаров.

In [ ]:
def cold_start_report(
    train_frame: pl.DataFrame,
    evaluation_frame: pl.DataFrame,
) -> dict[str, float]:
    # PUT YOUR CODE HERE
    return


tests.check_cold_start_report(cold_start_report)

In [ ]:
split_report = pd.DataFrame(
    {
        "validation": cold_start_report(train, validation),
        "test": cold_start_report(train_validation, test),
    }
).T
split_report

## 3. Метрики качества отбора кандидатов (2p)

Офлайн-метрики позволяют сравнивать модели до запуска онлайн-эксперимента. В задаче отбора кандидатов основной критерий — `Recall@K`: товар, не попавший в первые $K$ кандидатов, уже не сможет быть возвращен моделью ранжирования.

Каждую метрику реализуем отдельной функцией.
Все функции принимают `recommendations` и `targets` как списки списков. Элементы с одинаковым внешним индексом относятся к одному пользователю. Например, `recommendations[3]` и `targets[3]` описывают рекомендации и релевантные товары четвёртого пользователя. Поэтому длины внешних списков обязаны совпадать.

В реализациях намеренно используются обычные циклы: код должен буквально
повторять формулу и легко читаться.

### 3.1. Подготовка входов

1. `validate_metric_inputs` — проверка, что длины внешних списков совпадают,
   а $K$ положителен;
2. `build_target_lists` — списки релевантных товаров в порядке переданных `user_ids`:
   без дублей; незнакомый пользователь получает
   пустой список.


In [ ]:
NestedItems = Sequence[Sequence[str]]


def validate_metric_inputs(
    recommendations: NestedItems,
    targets: NestedItems,
    k: int,
) -> None:
    if len(recommendations) != len(targets):
        raise ValueError(
            "recommendations и targets должны содержать одинаковое число пользователей"
        )
    if k <= 0:
        raise ValueError("k должен быть положительным")


def build_target_lists(
    frame: pl.DataFrame,
    user_ids: Sequence[str],
) -> list[list[str]]:
    """Собирает позитивные взаимодействия в порядке переданных пользователей."""
    # PUT YOUR CODE HERE
    return


tests.check_metric_data_helpers(validate_metric_inputs, build_target_lists)

### 3.2. Recall@K

Для пользователя $u$ обозначим множество релевантных товаров через $G_u$, а первые
$K$ рекомендаций — через $R_u^K$. Полнота пользователя равна

$$Recall@K_u=\frac{|G_u\cap R_u^K|}{|G_u|}.$$

Числитель — число найденных релевантных товаров, знаменатель — число всех
релевантных товаров пользователя. Если $G_u$ пусто, значение пользователя считаем
равным нулю. Итоговый `Recall@K` — среднее по всем $n$ пользователям:

$$Recall@K=\frac{1}{n}\sum_{u=1}^{n}Recall@K_u.$$

---

Это главный критерий отбора кандидатов: товар, не попавший в первые $K$, модель
ранжирования уже не вернёт.

In [ ]:
def recall_at_k(
    recommendations: NestedItems,
    targets: NestedItems,
    k: int,
) -> float:
    validate_metric_inputs(recommendations, targets, k)
    if not recommendations:
        return 0.0

    # PUT YOUR CODE HERE
    return


tests.check_recall_at_k(recall_at_k)

### 3.3. HitRate@K

`HitRate@K` проверяет, есть ли у пользователя хотя бы одно попадание:

$$HitRate@K_u=\mathbb{1}[|G_u\cap R_u^K|>0],$$

где $\mathbb{1}[\cdot]$ — индикатор условия. Итоговая метрика — доля пользователей
хотя бы с одним релевантным товаром среди первых $K$ рекомендаций:

$$HitRate@K=\frac{1}{n}\sum_{u=1}^{n}HitRate@K_u.$$

---

В отличие от `Recall@K`, метрика не зависит от того, сколько у пользователя
релевантных товаров, и отвечает на простой продуктовый вопрос: какой доле
пользователей мы вообще угадали хоть что-то.

In [ ]:
def hit_rate_at_k(
    recommendations: NestedItems,
    targets: NestedItems,
    k: int,
) -> float:
    validate_metric_inputs(recommendations, targets, k)
    if not recommendations:
        return 0.0

    # PUT YOUR CODE HERE
    return


tests.check_hit_rate_at_k(hit_rate_at_k)

### 3.4. MAP@K

Пусть $rel_{ur}\in\{0,1\}$ показывает, релевантен ли товар пользователя $u$ на
позиции $r$. Точность префикса длины $r$ равна

$$P@r_u=\frac{\sum_{j=1}^{r}rel_{uj}}{r}.$$

Средняя точность одного пользователя учитывает точность только в позициях с
попаданиями:

$$AP@K_u=\frac{\sum_{r=1}^{K}P@r_u\,rel_{ur}}{\min(|G_u|,K)}.$$

Знаменатель — максимально возможное число попаданий в $K$ позиций. Для пустого
$G_u$ принимаем $AP@K_u=0$. `MAP@K` — среднее значение по пользователям:

$$MAP@K=\frac{1}{n}\sum_{u=1}^{n}AP@K_u.$$

---

`Recall@K` не различает, стоял релевантный товар первым или двадцатым. `MAP@K`
учитывает позиции всех попаданий сразу и потому чувствителен к порядку внутри выдачи.

In [ ]:
def map_at_k(
    recommendations: NestedItems,
    targets: NestedItems,
    k: int,
) -> float:
    validate_metric_inputs(recommendations, targets, k)
    if not recommendations:
        return 0.0

    # PUT YOUR CODE HERE
    return


tests.check_map_at_k(map_at_k)

### 3.5. MRR@K

`MRR@K` учитывает только первое попадание. Обозначим его позицию для пользователя
$u$ через

$$rank_u=\min\{r\leq K:rel_{ur}=1\}.$$

Обратный ранг равен $RR@K_u=1/rank_u$. Если попаданий среди первых $K$ позиций нет,
принимаем $RR@K_u=0$. Итоговая метрика усредняет обратные ранги:

$$MRR@K=\frac{1}{n}\sum_{u=1}^{n}RR@K_u.$$

Чем раньше встречается первый релевантный товар, тем ближе вклад пользователя к
единице.

---

Метрика описывает сценарий, в котором пользователю достаточно одного подходящего
товара: важно лишь, насколько высоко он стоит.

In [ ]:
def mrr_at_k(
    recommendations: NestedItems,
    targets: NestedItems,
    k: int,
) -> float:
    validate_metric_inputs(recommendations, targets, k)
    if not recommendations:
        return 0.0

    # PUT YOUR CODE HERE
    return


tests.check_mrr_at_k(mrr_at_k)

### 3.6. NDCG@K

`NDCG@K` учитывает все попадания и сильнее вознаграждает ранние. Для бинарной
релевантности

$$DCG@K_u=\sum_{r=1}^{K}\frac{rel_{ur}}{\log_2(r+1)}.$$

Здесь $rel_{ur}$ — индикатор релевантности товара на позиции $r$, а логарифм
уменьшает вклад поздних позиций. `IDCG@K` — значение `DCG@K` для идеальной выдачи, в
которой все релевантные товары стоят первыми. Нормированная метрика пользователя

$$NDCG@K_u=\frac{DCG@K_u}{IDCG@K_u}$$

равна нулю при $IDCG@K_u=0$. Итоговый `NDCG@K` — среднее по пользователям.

---

`MAP@K` штрафует поздние позиции линейно, `NDCG@K` — логарифмически, и нормировка на идеальную выдачу делает значения сравнимыми между пользователями с разным числом релевантных товаров.

In [ ]:
def ndcg_at_k(
    recommendations: NestedItems,
    targets: NestedItems,
    k: int,
) -> float:
    validate_metric_inputs(recommendations, targets, k)
    if not recommendations:
        return 0.0

    # PUT YOUR CODE HERE
    return


tests.check_ndcg_at_k(ndcg_at_k)

### 3.7. Coverage@K

`Coverage@K` измеряет, насколько широко модель использует множество товаров $I$:

$$Coverage@K=\frac{|(\bigcup_{u=1}^{n}R_u^K)\cap I|}{|I|}.$$

В числителе стоит число уникальных товаров, встретившихся хотя бы в одной выдаче, а
в знаменателе — число доступных товаров. Аргумент `n_items` задаёт $|I|$. Если он не
передан, множеством $I$ считаются уникальные товары из `targets`, а рекомендации вне
этого множества не увеличивают числитель.

---

Это единственная метрика в наборе, которая говорит не о точности, а о разнообразии. Она сразу показывает вырождение модели: если выдача всем одинакова, `Coverage@K` падает почти до нуля даже при хорошем `Recall@K`.

In [ ]:
def coverage_at_k(
    recommendations: NestedItems,
    targets: NestedItems,
    k: int,
    n_items: int | None = None,
) -> float:
    validate_metric_inputs(recommendations, targets, k)
    if n_items is not None and n_items <= 0:
        raise ValueError("n_items должен быть положительным или None")

    # PUT YOUR CODE HERE
    return


tests.check_coverage_at_k(coverage_at_k)

### 3.8. Единый вызов метрик

Функция `evaluate` для каждого значения $K$ из `cutoffs` вызывает метрики и складывает результаты в один словарь с ключами вида
`<metric_name>@<K>`.

In [ ]:
def evaluate(
    recommendations: NestedItems,
    targets: NestedItems,
    cutoffs: Sequence[int] = (20, 50),
    n_items: int | None = None,
) -> dict[str, float]:
    normalized_cutoffs = tuple(dict.fromkeys(map(int, cutoffs)))
    if not normalized_cutoffs or any(k <= 0 for k in normalized_cutoffs):
        raise ValueError("cutoffs должны содержать положительные значения")

    # PUT YOUR CODE HERE
    return


tests.check_evaluate(evaluate)

## 4. Общий интерфейс и экспериментальный контур (2p)

Все модели наследуются от одного базового класса.
Различаются они ровно одним — тем, как вычисляется оценка релевантности товара
для пользователя. Всё остальное — каталог, история, исключение просмотренных
товаров, отбор top-K и выдача дефолтных рекомендаций холодным пользователям — реализовано здесь один раз.

По порядку:

1. `top_n_indices` — быстрый отбор top-K из матрицы оценок;
2. `BaseRecommender` — интерфейс `fit`/`recommend` и общая статистика;
3. `MatrixRecommender` — модели поверх разреженной матрицы взаимодействий;
4. функции экспериментального контура: `validation` → выбор параметров → `test`.

### 4.1. Отбор top-K

Полный порядок всех товаров нам никогда не нужен — нужны только первые $K$. Полная
сортировка стоит $O(|I|\log|I|)$ на пользователя, а `np.argpartition` находит $K$
наибольших оценок за $O(|I|)$; сортировать приходится только эти $K$ значений.

Нужно реализовать функцию `top_n_indices(scores, n)`. На вход — матрица оценок формы
`(число пользователей, |I|)`, на выход — индексы $n$ лучших товаров в каждой строке,
упорядоченные по убыванию оценки. Если $n$ больше числа столбцов, вернуть все столбцы.

Имейте в виду, что argpartition ставит `n` наибольших оценок в начало, но не упорядочивает их.

In [ ]:
def top_n_indices(scores: NDArray[np.float32], n: int) -> NDArray[np.int64]:
    # PUT YOUR CODE HERE
    return


tests.check_top_n_indices(top_n_indices)

### 4.2. Базовый класс (1p)

`BaseRecommender` — общий интерфейс всех моделей.

1. `fit` запоминает то, что нужно любой модели, и передаёт управление в `_fit`:
   * `items` — каталог в порядке убывания популярности; он же `popular_items`,
     то есть резервная выдача для пользователей без истории;
   * `item_to_index` — номер товара в этом каталоге;
   * `seen_items` — история пользователя, которую не нужно рекомендовать повторно.
2. `recommend` проверяет, что модель обучена и `topn` положителен, и возвращает по
   одному списку товаров на каждого пользователя, сохраняя порядок `user_ids`.
3. Три вспомогательных метода:
   * `_remove_seen` работает с уже упорядоченным списком товаров (по популярности
     или в случайном порядке) и прекращает обход, как только набрано `topn` товаров;
   * `_seen_indices` возвращает номера просмотренных товаров, чтобы модель могла
     проставить им `-inf` прямо в матрице оценок;
   * `_rank_by_scores` превращает матрицу оценок в списки идентификаторов товаров.


In [ ]:
class BaseRecommender(ABC):
    """Общий интерфейс рекомендательной модели."""

    name: str
    fitted: bool
    items: list[str]
    item_to_index: dict[str, int]
    popular_items: list[str]
    seen_items: dict[str, set[str]]
    show_progress: bool
    batch_size: int = 1024

    def __init__(self, name: str, show_progress: bool = True) -> None:
        self.name = name
        self.show_progress = show_progress
        self.fitted = False

    def fit(self, interactions: pl.DataFrame, **kwargs: Any) -> Self:
        """Запоминает каталог и историю, затем обучает конкретную модель."""
        popularity = (
            interactions.group_by("item_id")
            .agg(pl.len().alias("n_events"))
            .sort(["n_events", "item_id"], descending=[True, False])
        )
        # Каталог сразу хранится по убыванию популярности: это и перечень товаров,
        # и готовая резервная выдача для пользователей без истории.
        self.items = popularity["item_id"].to_list()
        self.item_to_index = {item: index for index, item in enumerate(self.items)}
        self.popular_items = self.items

        seen = (
            interactions.select(["user_id", "item_id"])
            .unique(maintain_order=True)
            .group_by("user_id", maintain_order=True)
            .agg(pl.col("item_id"))
        )
        self.seen_items = {
            user: set(item_list)
            for user, item_list in zip(
                seen["user_id"].to_list(), seen["item_id"].to_list()
            )
        }
        self._fit(interactions, **kwargs)
        self.fitted = True
        return self

    @abstractmethod
    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        """Обучает конкретную модель."""

    def recommend(self, user_ids: Sequence[str], topn: int = 50) -> list[list[str]]:
        """Возвращает списки рекомендаций в порядке user_ids."""
        if not self.fitted:
            raise ValueError("Сначала вызовите fit")
        if topn <= 0:
            raise ValueError("topn должен быть положительным")
        recommendations = self._recommend([str(user) for user in user_ids], topn)
        if len(recommendations) != len(user_ids):
            raise RuntimeError("Число списков рекомендаций не совпало с числом пользователей")
        return recommendations

    @abstractmethod
    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        """Строит рекомендации конкретной модели."""

    def _remove_seen(
        self,
        user_id: str,
        candidates: Iterable[str],
        topn: int,
    ) -> list[str]:
        """Берёт первые topn кандидатов, пропуская товары из истории пользователя."""
        # PUT YOUR CODE HERE
        return

    def _seen_indices(
        self,
        user_id: str,
        index: Mapping[str, int] | None = None,
    ) -> NDArray[np.int64]:
        """Номера просмотренных товаров в item_to_index или в переданном отображении."""
        # PUT YOUR CODE HERE
        return

    def _rank_by_scores(
        self,
        scores: NDArray[np.float32],
        vocabulary: Sequence[str],
        topn: int,
    ) -> list[list[str]]:
        """Переводит матрицу оценок в списки товаров; скрытые (-inf) отбрасываются.
        Испольузет top_n_indices."""
        # PUT YOUR CODE HERE
        return

    def _recommend_in_batches(
        self,
        user_ids: list[str],
        topn: int,
        is_known: Callable[[str], bool],
        score_known: Callable[[list[str], int], list[list[str]]],
    ) -> list[list[str]]:
        """Общий батч-цикл инференса: пользователи режутся на батчи размера
        batch_size, для известных вызывается score_known, для остальных —
        резервная выдача по popular_items. Этим циклом пользуются Random-,
        Matrix- и Transformer-модели, чтобы не повторять его реализацию трижды."""
        result: list[list[str]] = []
        starts = range(0, len(user_ids), self.batch_size)
        for start in tqdm(starts, desc=self.name, disable=not self.show_progress):
            batch = user_ids[start : start + self.batch_size]
            known = [user for user in batch if is_known(user)]
            ranked = iter(score_known(known, topn))
            for user in batch:
                result.append(
                    next(ranked)
                    if is_known(user)
                    else self._remove_seen(user, self.popular_items, topn)
                )
        return result


tests.check_base_recommender(BaseRecommender)

### 4.3. Базовый класс для матричных моделей (1p)

EASE, ALS, BPR и контентная модель работают с одной разреженной матрицей
$X\in\mathbb{R}^{|U|\times|I|}$, где строка — пользователь, столбец — товар.

`MatrixRecommender` наследуется от `BaseRecommender`, добавляя следующие методы:

1. `_create_interaction_matrix` — создает CSR-матрицу и отображения идентификаторов.
   Параметр `weight_column` позволяет записать в ячейки не единицы, а, например,
   оценку. `csr_matrix` складывает повторные пары $(u, i)$!
2. `_recommend` — общая процедура выдачи:
   * пользователи разбиваются на батчи, чтобы плотная матрица оценок не занимала
     слишком много памяти;
   * `_score_batch` даёт оценки всех товаров для батча;
   * просмотренным товарам проставляется `-inf`;
   * `top_n_indices` отбирает $K$ лучших;
   * пользователи, которых не было в обучении, получают популярные товары.
3. `_score_batch` — абстрактный метод: его реализует каждая конкретная модель.


In [ ]:
class MatrixRecommender(BaseRecommender):
    """Модели поверх разреженной матрицы взаимодействий."""

    users: list[str]
    user_to_id: dict[str, int]
    item_to_id: dict[str, int]
    id_to_item: list[str]
    interaction_matrix: sparse.csr_matrix

    def _create_interaction_matrix(
        self,
        interactions: pl.DataFrame,
        weight_column: str | None = None,
    ) -> None:
        """Строит CSR-матрицу взаимодействий и отображения идентификаторов."""
        self.users = interactions["user_id"].unique(maintain_order=True).to_list()
        self.user_to_id = ... # PUT YOUR CODE HERE
        self.id_to_item = ... # PUT YOUR CODE HERE
        self.item_to_id = ... # PUT YOUR CODE HERE

        # Векторизованный перевод id -> индекс: join с маленькой таблицей-словарём
        # вместо поэлементного .map(dict) по всему датасету.
        user_vocab = pl.DataFrame(
            {"user_id": self.users, "user_index": np.arange(len(self.users))}
        )
        item_vocab = pl.DataFrame(
            {"item_id": self.id_to_item, "item_index": np.arange(len(self.id_to_item))}
        )
        indexed = interactions.join(user_vocab, on="user_id", how="left").join(
            item_vocab, on="item_id", how="left"
        )
        rows = indexed["user_index"].to_numpy()
        columns = indexed["item_index"].to_numpy()
        values = (
            np.ones(len(interactions), dtype=np.float32)
            if weight_column is None
            else indexed[weight_column].to_numpy().astype(np.float32)
        )
        matrix = sparse.csr_matrix(
            (values, (rows, columns)),
            shape=(len(self.users), len(self.id_to_item)),
        )
        if weight_column is None:
            matrix.data[:] = 1.0
        self.interaction_matrix = matrix

    @abstractmethod
    def _score_batch(self, user_indices: NDArray[np.int64]) -> NDArray[np.float32]:
        """Возвращает оценки всех товаров матрицы для переданных пользователей."""

    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        """Батчами считает оценки; холодные пользователи получают популярные товары."""
        return self._recommend_in_batches(
            user_ids, topn, lambda user: user in self.user_to_id, self._rank_known
        )

    def _rank_known(self, user_ids: list[str], topn: int) -> list[list[str]]:
        """Считает оценки для известных пользователей и отбирает top-K."""
        if not user_ids:
            return []
        user_indices = np.array(
            [self.user_to_id[user] for user in user_ids], dtype=np.int64
        )
        scores = np.asarray(self._score_batch(user_indices), dtype=np.float32)
        for row, user in enumerate(user_ids):
            scores[row, self._seen_indices(user, self.item_to_id)] = -np.inf
        return self._rank_by_scores(scores, self.id_to_item, topn)


tests.check_matrix_recommender(MatrixRecommender)

### 4.4. Подбор гиперпараметров и финальная оценка

Тут реализуем протокол обучения и тестирования моделей. Потребуются следующие функции:

1. `evaluate_model` — обучает модель на одной выборке, получает `candidate_k` кандидатов для
   пользователей следующего периода и считает метрики. Возвращает тройку
   `(модель, метрики, рекомендации)`; в метрики добавляются `fit_seconds` и
   `inference_seconds`.
2. `validation_search` — таблица метрик по всей сетке параметров, отсортированная по `Recall@20`.
3. `select_best_params` — параметры строки с максимальным значением метрики
   (по умолчанию `Recall@20`).
4. `fit_best_on_test` — обучение лучшей конфигурации на `train + validation` и
   единственный расчёт метрик на `test`.

Один и тот же контур для всех восьми семейств моделей делает сравнение честным. И он же гарантирует главное правило: `test` используется ровно один раз, уже после того, как параметры выбраны.

In [ ]:
ModelFactory = Callable[[Mapping[str, Any]], BaseRecommender]


def evaluate_model(
    model: BaseRecommender,
    fit_frame: pl.DataFrame,
    evaluation_frame: pl.DataFrame,
    candidate_k: int = 50,
    metric_cutoffs: Sequence[int] = (20, 50),
    fit_kwargs: Mapping[str, Any] | None = None,
) -> tuple[BaseRecommender, dict[str, float], list[list[str]]]:
    """Обучает модель и вычисляет метрики на следующем периоде."""
    started = time.perf_counter()
    model.fit(fit_frame, **dict(fit_kwargs or {}))
    fit_seconds = time.perf_counter() - started

    user_ids = evaluation_frame["user_id"].unique(maintain_order=True).to_list()
    targets = build_target_lists(evaluation_frame, user_ids)

    started = time.perf_counter()
    recommendations = model.recommend(user_ids, topn=candidate_k)
    inference_seconds = time.perf_counter() - started

    metrics = evaluate(
        recommendations,
        targets,
        cutoffs=metric_cutoffs,
        n_items=fit_frame["item_id"].n_unique(),
    )
    metrics = {
        **metrics,
        "fit_seconds": fit_seconds,
        "inference_seconds": inference_seconds,
    }
    return model, metrics, recommendations


def validation_search(
    family: str,
    parameter_grid: Sequence[Mapping[str, Any]],
    factory: ModelFactory,
    train_frame: pl.DataFrame,
    validation_frame: pl.DataFrame,
    fit_kwargs: Mapping[str, Any] | None = None,
) -> pd.DataFrame:
    """Строит таблицу validation-метрик для сетки параметров."""
    rows: list[dict[str, Any]] = []
    for config_id, params in enumerate(
        tqdm(parameter_grid, desc=f"{family}: validation"), start=1
    ):
        _, metrics, _ = evaluate_model(
            factory(params), train_frame, validation_frame, fit_kwargs=fit_kwargs
        )
        rows.append(
            {"family": family, "config_id": config_id, "params": dict(params), **metrics}
        )
    return (
        pd.DataFrame(rows)
        .sort_values(
            ["recall@20", "ndcg@20", "config_id"],
            ascending=[False, False, True],
            kind="stable",
        )
        .reset_index(drop=True)
    )


def select_best_params(
    validation_results: pd.DataFrame,
    metric: str = "recall@20",
) -> dict[str, Any]:
    """Возвращает параметры конфигурации с максимальным значением метрики."""
    if validation_results.empty:
        raise ValueError("Таблица validation-результатов пуста")
    best = validation_results.sort_values(
        [metric, "config_id"], ascending=[False, True], kind="stable"
    ).iloc[0]
    return dict(best["params"])


def fit_best_on_test(
    family: str,
    best_params: Mapping[str, Any],
    factory: ModelFactory,
    train_validation_frame: pl.DataFrame,
    test_frame: pl.DataFrame,
    fit_kwargs: Mapping[str, Any] | None = None,
) -> tuple[BaseRecommender, dict[str, Any], list[list[str]]]:
    """Обучает лучшую конфигурацию на train+validation и оценивает test."""
    model, metrics, recommendations = evaluate_model(
        factory(best_params),
        train_validation_frame,
        test_frame,
        fit_kwargs=fit_kwargs,
    )
    result = {"family": family, "params": dict(best_params), **metrics}
    return model, result, recommendations


VALIDATION_RESULTS: dict[str, pd.DataFrame] = {}
BEST_MODELS: dict[str, BaseRecommender] = {}
METRICS: dict[str, dict[str, Any]] = {}

## 5. Random (1p)

Random задаёт нижнюю границу качества и помогает проверить корректность конвейера
оценки. Каждой паре пользователя и товара присваивается псевдослучайная оценка
релевантности; фиксированное начальное состояние генератора делает результат
воспроизводимым.

### 5.1. Модель отбора кандидатов

**Что нужно реализовать**

`RandomRetriever`: генерировать псевдослучайные оценки сразу для целого батча
пользователей, товарам из истории проставлять `-inf` и передавать результат в
`_rank_by_scores`. Генератор создаётся из `seed` при каждом вызове `_recommend`,
чтобы выдача была воспроизводимой.

In [ ]:
class RandomRetriever(BaseRecommender):
    """Случайно выбирает товары, которых не было в истории пользователя."""

    seed: int

    def __init__(self, seed: int = 42, show_progress: bool = True) -> None:
        super().__init__("Random", show_progress)
        self.seed = seed

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        """Random не обучается: каталог уже собран в fit."""

    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        generator = np.random.default_rng(self.seed)
        # PUT YOUR CODE HERE
        return


tests.check_random_retriever(RandomRetriever)

### 5.2. Подбор гиперпараметров

Запустим модель с разными `seed`-ами (действительно, а какие тут гиперпараметры?)

In [ ]:
random_model_grid: list[dict[str, Any]] = [{"seed": 17}, {"seed": 42}, {"seed": 101}]


def make_random_model(params: Mapping[str, Any]) -> BaseRecommender:
    return RandomRetriever(seed=int(params["seed"]))


random_model_validation = validation_search(
    family="Random",
    parameter_grid=random_model_grid,
    factory=make_random_model,
    train_frame=train,
    validation_frame=validation,
)
VALIDATION_RESULTS["Random"] = random_model_validation
display(random_model_validation)

### 5.3. Финальное обучение


In [ ]:
random_model_best_params = select_best_params(random_model_validation)
random_model_best_model, random_model_test_result, _ = fit_best_on_test(
    family="Random",
    best_params=random_model_best_params,
    factory=make_random_model,
    train_validation_frame=train_validation,
    test_frame=test,
)
BEST_MODELS["Random"] = random_model_best_model
METRICS["Random"] = random_model_test_result
display(pd.DataFrame([random_model_test_result]))

## 6. TopPopular (1p)

TopPopular не персонализирует выдачу, но часто оказывается сильной базовой моделью
из-за смещения данных в сторону популярных товаров. Пусть $E$ — множество событий,
$i_e$ — товар в событии $e$, а $rating_e$ — его оценка. Для режима `count`

$$s_i=|\{e\in E:i_e=i\}|,$$

то есть $s_i$ равно числу взаимодействий с товаром $i$. Для режима `rating`

$$s_i=\sum_{e\in E:i_e=i}rating_e,$$

поэтому частые события с высокой оценкой дают товару больший итоговый вес.

### 6.1. Модель отбора кандидатов

**Что нужно реализовать**

`TopPopularRetriever`: в `_fit` посчитать общий для всех пользователей порядок
товаров (по числу событий или по сумме оценок, в зависимости от `weighting`), в
`_recommend` убрать из него историю конкретного пользователя (используйте `_remove_seen`).


In [ ]:
class TopPopularRetriever(BaseRecommender):
    """Сортирует товары по числу событий или по сумме оценок."""

    weighting: Literal["count", "rating"]
    ranked_items: list[str]

    def __init__(
        self,
        weighting: Literal["count", "rating"] = "count",
        show_progress: bool = True,
    ) -> None:
        super().__init__("TopPopular", show_progress)
        self.weighting = weighting

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        # PUT YOUR CODE HERE
        return

    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        # PUT YOUR CODE HERE
        return


tests.check_top_popular_retriever(TopPopularRetriever)

### 6.2. Подбор гиперпараметров

Посмотрим, как себя ведут разныет типы взвешивания.

In [ ]:
top_popular_grid: list[dict[str, Any]] = [
    {"weighting": "count"},
    {"weighting": "rating"},
]


def make_top_popular(params: Mapping[str, Any]) -> BaseRecommender:
    weighting = cast(Literal["count", "rating"], params["weighting"])
    return TopPopularRetriever(weighting=weighting)


top_popular_validation = validation_search(
    family="TopPopular",
    parameter_grid=top_popular_grid,
    factory=make_top_popular,
    train_frame=train,
    validation_frame=validation,
)
VALIDATION_RESULTS["TopPopular"] = top_popular_validation
display(top_popular_validation)

### 6.3. Финальное обучение

In [ ]:
top_popular_best_params = select_best_params(top_popular_validation)
top_popular_best_model, top_popular_test_result, _ = fit_best_on_test(
    family="TopPopular",
    best_params=top_popular_best_params,
    factory=make_top_popular,
    train_validation_frame=train_validation,
    test_frame=test,
)
BEST_MODELS["TopPopular"] = top_popular_best_model
METRICS["TopPopular"] = top_popular_test_result
display(pd.DataFrame([top_popular_test_result]))

## 7. DecayPop (1p)

Обычная популярность одинаково учитывает старые и новые события. В динамичном
каталоге полезнее экспоненциальное затухание

$$s_i=\sum_{e\in E:i_e=i}rating_e\,2^{-\Delta t_e/h}.$$

Здесь $E$ — множество событий в `train`, $i_e$ и $rating_e$ — товар и оценка в
событии $e$, $\Delta t_e$ — возраст события в днях относительно последнего события
в `train`, а $h$ — период полураспада `half_life_days`. Через $h$ дней вклад события
уменьшается вдвое; чем меньше $h$, тем быстрее модель забывает прошлое.

### 7.1. Модель отбора кандидатов

**Что нужно реализовать**

`DecayPopularityRetriever`: вычислить возраст каждого события в днях относительно
последнего события выборки, взвесить оценки множителем $2^{-\Delta t_e/h}$ и
просуммировать веса по товарам. Выдача устроена так же, как в TopPopular.


In [ ]:
class DecayPopularityRetriever(BaseRecommender):
    """Учитывает оценку события и экспоненциальное затухание по времени."""

    half_life_days: float
    ranked_items: list[str]

    def __init__(self, half_life_days: float = 90.0, show_progress: bool = True) -> None:
        super().__init__("DecayPopularity", show_progress)
        self.half_life_days = half_life_days

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        # PUT YOUR CODE HERE
        return

    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        # PUT YOUR CODE HERE
        return


tests.check_decay_popularity_retriever(DecayPopularityRetriever)

### 7.2. Подбор гиперпараметров


In [ ]:
decay_popularity_grid: list[dict[str, Any]] = [
    {"half_life_days": value} for value in [30.0, 90.0, 365.0]
]


def make_decay_popularity(params: Mapping[str, Any]) -> BaseRecommender:
    return DecayPopularityRetriever(half_life_days=float(params["half_life_days"]))


decay_popularity_validation = validation_search(
    family="DecayPopularity",
    parameter_grid=decay_popularity_grid,
    factory=make_decay_popularity,
    train_frame=train,
    validation_frame=validation,
)
VALIDATION_RESULTS["DecayPopularity"] = decay_popularity_validation
display(decay_popularity_validation)

### 7.3. Финальное обучение

In [ ]:
decay_popularity_best_params = select_best_params(decay_popularity_validation)
decay_popularity_best_model, decay_popularity_test_result, _ = fit_best_on_test(
    family="DecayPopularity",
    best_params=decay_popularity_best_params,
    factory=make_decay_popularity,
    train_validation_frame=train_validation,
    test_frame=test,
)
BEST_MODELS["DecayPopularity"] = decay_popularity_best_model
METRICS["DecayPopularity"] = decay_popularity_test_result
display(pd.DataFrame([decay_popularity_test_result]))

## 8. EASE (1p)

EASE — линейная модель связей между товарами. Пусть
$X\in\{0,1\}^{|U|\times|I|}$ — бинарная матрица взаимодействий пользователей $U$ с
товарами $I$. Матрица

$$G=X^TX+\lambda I_{|I|}$$

содержит совместную встречаемость товаров, а слагаемое $\lambda I_{|I|}$ добавляет
регуляризацию; $I_{|I|}$ — единичная матрица размера $|I|$, а $\lambda>0$ —
коэффициент регуляризации. Решив $GP=I_{|I|}$, получаем $P=G^{-1}$, после чего

$$B_{ij}=-\frac{P_{ij}}{P_{jj}},\qquad B_{jj}=0.$$

Коэффициент $B_{ij}$ описывает вклад уже просмотренного товара $i$ в оценку
релевантности кандидата $j$; нулевая диагональ запрещает модели рекомендовать
товар на основании его самого. Если $X_u$ — строка истории пользователя $u$, то
оценки всех кандидатов равны $s_u=X_uB$. Параметр $\lambda$ уменьшает переобучение
и улучшает обусловленность системы.

### 8.1. Замкнутое решение

**Что нужно реализовать**

Функцию `fit_ease_weights(matrix, regularization)`: собрать $G=X^TX+\lambda I$,
решить $GP=I$, получить $B_{ij}=-P_{ij}/P_{jj}$ и обнулить диагональ. Используйте `np.linalg.solve(G, I)`, а не явный вызов `inv`: решение линейной
системы численно устойчивее. Считать нужно в `float64` — обращение плохо
обусловленной матрицы чувствительно к округлению.

In [ ]:
def fit_ease_weights(
    matrix: sparse.csr_matrix,
    regularization: float,
) -> NDArray[np.float64]:
    # PUT YOUR CODE HERE
    return


tests.check_ease_weights(fit_ease_weights)

### 8.2. Модель отбора кандидатов

**Что нужно реализовать**

`EASERetriever`: в `_fit` отобрать `max_items` самых частых товаров, построить по ним
матрицу взаимодействий и решить систему; в `_score_batch` вернуть $s_u=X_uB$. Матрица $B$ плотная и имеет размер $|I|\times|I|$, поэтому по всему каталогу её
считать дорого. Всё остальное — батчи, маскирование истории, top-K — уже реализовано
в `MatrixRecommender`.

In [ ]:
class EASERetriever(MatrixRecommender):
    """Линейная модель связей между товарами EASE."""

    regularization: float
    max_items: int
    weights: NDArray[np.float64]

    def __init__(
        self,
        regularization: float = 300.0,
        max_items: int = 2_000,
        show_progress: bool = True,
    ) -> None:
        super().__init__("EASE", show_progress)
        self.regularization = regularization
        self.max_items = max_items

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        # PUT YOUR CODE HERE
        return

    def _score_batch(self, user_indices: NDArray[np.int64]) -> NDArray[np.float32]:
        """s_u = X_u B."""
        # PUT YOUR CODE HERE
        return


tests.check_ease_retriever(EASERetriever)

### 8.3. Подбор гиперпараметров


In [ ]:
ease_grid: list[dict[str, Any]] = [
    {"regularization": value} for value in [50.0, 300.0, 1_000.0]
]


def make_ease(params: Mapping[str, Any]) -> BaseRecommender:
    return EASERetriever(regularization=float(params["regularization"]))


ease_validation = validation_search(
    family="EASE",
    parameter_grid=ease_grid,
    factory=make_ease,
    train_frame=train,
    validation_frame=validation,
)
VALIDATION_RESULTS["EASE"] = ease_validation
display(ease_validation)

### 8.4. Финальное обучение

In [ ]:
ease_best_params = select_best_params(ease_validation)
ease_best_model, ease_test_result, _ = fit_best_on_test(
    family="EASE",
    best_params=ease_best_params,
    factory=make_ease,
    train_validation_frame=train_validation,
    test_frame=test,
)
BEST_MODELS["EASE"] = ease_best_model
METRICS["EASE"] = ease_test_result
display(pd.DataFrame([ease_test_result]))

## 9. ALS (1p)

ALS для неявного отклика (implicit feedback) представляет пользователя $u$
вектором $x_u$, товар $i$ — вектором $y_i$ и минимизирует

$$
\sum_{u\in U,\,i\in I}
c_{ui}(p_{ui}-x_u^T y_i)^2
+
\lambda\left(
\sum_{u\in U}\|x_u\|^2+
\sum_{i\in I}\|y_i\|^2
\right).
$$

Здесь $U$ и $I$ — множества пользователей и товаров;
$p_{ui}=1$ для наблюдаемого взаимодействия и $0$ иначе.
Уверенность в наблюдении задаётся как

$$
c_{ui}=1+\alpha r_{ui},
$$

где $r_{ui}\geq 0$ — сила наблюдаемого взаимодействия, а $\alpha$ управляет
вкладом наблюдений в функцию потерь.

### 9.1. Модель отбора кандидатов

**Что нужно реализовать**

`ALSRetriever`: в `_fit` построить бинарную матрицу взаимодействий и обучить
`implicit.als.AlternatingLeastSquares`; в `_score_batch` вернуть $s_{ui}=x_u^T y_i$.

Важная деталь: API библиотеки `implicit` отличается от обозначений выше.
Ненулевые значения sparse-матрицы, переданной в `fit`, интерпретируются как
confidence наблюдаемых взаимодействий, а для ненаблюдаемых пар библиотека
неявно использует $c_{ui}=1$. Параметр `alpha` конструктора дополнительно
умножает ненулевые значения переданной матрицы.

Поэтому если передать бинарную матрицу и установить `alpha=a`, то для
наблюдаемых пар библиотека будет использовать $c_{ui}=a$, а не $1+a$.

Чтобы реализовать определение

$$
c_{ui}=1+\alpha r_{ui},
$$

для бинарной матрицы взаимодействий можно либо заранее записать в её ненулевые
элементы значения $1+\alpha$ и обучать модель с `alpha=1.0`, либо эквивалентно
передать бинарную матрицу и использовать в конструкторе библиотеки
`alpha`$=1 + a$.

In [ ]:
class ALSRetriever(MatrixRecommender):
    """Обёртка над implicit ALS."""

    factors: int
    regularization: float
    alpha: float
    iterations: int
    seed: int
    model: Any

    def __init__(
        self,
        factors: int = 64,
        regularization: float = 0.01,
        alpha: float = 1.0,
        iterations: int = 15,
        seed: int = 42,
        show_progress: bool = True,
    ) -> None:
        super().__init__("ALS", show_progress)
        self.factors = factors
        self.regularization = regularization
        self.alpha = alpha
        self.iterations = iterations
        self.seed = seed

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        # PUT YOUR CODE HERE
        return

    def _score_batch(self, user_indices: NDArray[np.int64]) -> NDArray[np.float32]:
        """s_ui = x_u^T y_i."""
        # PUT YOUR CODE HERE
        return


tests.check_als_retriever(ALSRetriever)

### 9.2. Подбор гиперпараметров


In [ ]:
als_grid: list[dict[str, Any]] = [
    {
        "factors": factors,
        "regularization": regularization,
        "alpha": alpha,
        "iterations": 15,
    }
    for factors in [32, 64]
    for regularization in [0.01, 0.1]
    for alpha in [1.0]
]


def make_als(params: Mapping[str, Any]) -> BaseRecommender:
    return ALSRetriever(
        factors=int(params["factors"]),
        regularization=float(params["regularization"]),
        alpha=float(params["alpha"]),
        iterations=int(params["iterations"]),
    )


als_validation = validation_search(
    family="ALS",
    parameter_grid=als_grid,
    factory=make_als,
    train_frame=train,
    validation_frame=validation,
)
VALIDATION_RESULTS["ALS"] = als_validation
display(als_validation)

### 9.3. Финальное обучение

In [ ]:
als_best_params = select_best_params(als_validation)
als_best_model, als_test_result, _ = fit_best_on_test(
    family="ALS",
    best_params=als_best_params,
    factory=make_als,
    train_validation_frame=train_validation,
    test_frame=test,
)
BEST_MODELS["ALS"] = als_best_model
METRICS["ALS"] = als_test_result
display(pd.DataFrame([als_test_result]))

## 10. BPR (1p)

BPR оптимизирует относительный порядок. Для пользователя $u$, наблюдаемого товара
$i^+$ и случайно выбранного отрицательного товара $i^-$ минимизируется

$$-\log\sigma(x_u^Ty_{i^+}-x_u^Ty_{i^-})+\lambda\|\Theta\|^2.$$

Здесь $x_u$, $y_{i^+}$ и $y_{i^-}$ — латентные векторы пользователя и товаров;
разность их скалярных произведений должна стать положительной; $\sigma$ —
сигмоидальная функция; $\Theta$ — совокупность всех обучаемых векторов; $\lambda$ —
коэффициент регуляризации. `factors` задаёт размерность векторов, а качество также
зависит от способа выбора отрицательных примеров.

### 10.1. Модель отбора кандидатов

**Что нужно реализовать**

`BPRRetriever`: в `_fit` построить матрицу взаимодействий и обучить
`implicit.bpr.BayesianPersonalizedRanking`; в `_score_batch` вернуть
$s_{ui}=x_u^Ty_i$ — формула оценки та же, что у ALS.

BPR смотрит только на факт взаимодействия и оптимизирует относительный порядок,
поэтому веса уверенности ему не нужны: `alpha` в конструктор не передаётся.

In [ ]:
class BPRRetriever(MatrixRecommender):
    """Обёртка над Bayesian Personalized Ranking."""

    factors: int
    regularization: float
    iterations: int
    seed: int
    model: Any

    def __init__(
        self,
        factors: int = 64,
        regularization: float = 0.01,
        iterations: int = 100,
        seed: int = 42,
        show_progress: bool = True,
    ) -> None:
        super().__init__("BPR", show_progress)
        self.factors = factors
        self.regularization = regularization
        self.iterations = iterations
        self.seed = seed

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        # PUT YOUR CODE HERE
        return

    def _score_batch(self, user_indices: NDArray[np.int64]) -> NDArray[np.float32]:
        """s_ui = x_u^T y_i."""
        # PUT YOUR CODE HERE
        return


tests.check_bpr_retriever(BPRRetriever)

### 10.2. Подбор гиперпараметров

In [ ]:
bpr_grid: list[dict[str, Any]] = [
    {"factors": factors, "regularization": regularization, "iterations": 100}
    for factors in [32, 64]
    for regularization in [0.01, 0.1]
]


def make_bpr(params: Mapping[str, Any]) -> BaseRecommender:
    return BPRRetriever(
        factors=int(params["factors"]),
        regularization=float(params["regularization"]),
        iterations=int(params["iterations"]),
    )


bpr_validation = validation_search(
    family="BPR",
    parameter_grid=bpr_grid,
    factory=make_bpr,
    train_frame=train,
    validation_frame=validation,
)
VALIDATION_RESULTS["BPR"] = bpr_validation
display(bpr_validation)

### 10.3. Финальное обучение


In [ ]:
bpr_best_params = select_best_params(bpr_validation)
bpr_best_model, bpr_test_result, _ = fit_best_on_test(
    family="BPR",
    best_params=bpr_best_params,
    factory=make_bpr,
    train_validation_frame=train_validation,
    test_frame=test,
)
BEST_MODELS["BPR"] = bpr_best_model
METRICS["BPR"] = bpr_test_result
display(pd.DataFrame([bpr_test_result]))

## 11. Контентная модель ItemKNN (1p)

Коллаборативные модели ничего не знают о содержании самих товаров. Контентная
модель кодирует название товара $i$ нормированным вектором $e_i$. Пусть $H_u$ —
множество товаров в истории пользователя $u$. Его профиль определяется как
взвешенное среднее

$$p_u=\frac{\sum_{i\in H_u}w_{ui}e_i}{\sum_{i\in H_u}w_{ui}},$$

где при `uniform` вес $w_{ui}=1$, а при `rating` он равен оценке пользователя
$rating_{ui}$. Оценка релевантности кандидата — косинусная близость

$$s_{ui}=\frac{p_u^Te_i}{\|p_u\|\,\|e_i\|}.$$

Здесь числитель — скалярное произведение профиля и вектора товара, а знаменатель —
произведение их евклидовых норм. После L2-нормировки обоих векторов косинусная
близость вычисляется одним матричным умножением.

### 11.1. Векторные представления названий

**Что нужно реализовать**

Функцию `encode_titles`: прогнать названия товаров через `SentenceTransformer.encode`
и нормировать полученные векторы. Аргумент `encoder` позволяет передать уже
загруженную модель.

Обратите внимание, что после L2-нормировки косинусная близость превращается в обычное скалярное
произведение.

In [ ]:
def encode_titles(
    titles: Sequence[str],
    model_name: str,
    encoder: Any | None = None,
    show_progress: bool = True,
) -> NDArray[np.float32]:
    """Кодирует названия товаров и нормирует векторы."""
    text_encoder = encoder or SentenceTransformer(model_name)
    # PUT YOUR CODE HERE
    return


tests.check_encode_titles(encode_titles)

### 11.2. Профили пользователей и отбор кандидатов

**Что нужно реализовать**

`TextItemKNNRetriever`: в `_fit` построить матрицу взаимодействий (с
`weight_column="rating"` для режима `rating`), закодировать названия товаров и
получить профили одним умножением матрицы на матрицу векторов; в `_score_batch`
вернуть косинусную близость профиля и вектора товара.

Строка $u$ произведения равна $\sum_{i\in H_u}w_{ui}e_i$, а деление на сумму весов
заменяется L2-нормировкой — она не меняет порядок косинусных близостей. Так вся
формула профиля укладывается в одно матричное умножение вместо цикла по
пользователям.

In [ ]:
class TextItemKNNRetriever(MatrixRecommender):
    """Сравнивает профиль пользователя с векторами названий товаров."""

    weighting: Literal["uniform", "rating"]
    model_name: str
    encoder: Any | None
    item_embeddings: NDArray[np.float32]
    user_profiles: NDArray[np.float32]

    def __init__(
        self,
        weighting: Literal["uniform", "rating"] = "rating",
        model_name: str = "sentence-transformers/paraphrase-MiniLM-L3-v2",
        encoder: Any | None = None,
        show_progress: bool = True,
    ) -> None:
        super().__init__("ContentItemKNN", show_progress)
        self.weighting = weighting
        self.model_name = model_name
        self.encoder = encoder

    def _fit(
        self,
        interactions: pl.DataFrame,
        *,
        items: pl.DataFrame,
        **kwargs: Any,
    ) -> None:
        # PUT YOUR CODE HERE
        return

    def _score_batch(self, user_indices: NDArray[np.int64]) -> NDArray[np.float32]:
        """s_ui = cos(p_u, e_i) — оба вектора уже нормированы."""
        # PUT YOUR CODE HERE
        return


tests.check_text_itemknn(TextItemKNNRetriever)

### 11.3. Подбор гиперпараметров

In [ ]:
itemknn_grid: list[dict[str, Any]] = [{"weighting": "uniform"}, {"weighting": "rating"}]
title_encoder = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2")


def make_itemknn(params: Mapping[str, Any]) -> BaseRecommender:
    return TextItemKNNRetriever(
        weighting=cast(Literal["uniform", "rating"], params["weighting"]),
        encoder=title_encoder,
    )


itemknn_validation = validation_search(
    family="ContentItemKNN",
    parameter_grid=itemknn_grid,
    factory=make_itemknn,
    train_frame=train,
    validation_frame=validation,
    fit_kwargs={"items": items},
)
VALIDATION_RESULTS["ContentItemKNN"] = itemknn_validation
display(itemknn_validation)

### 11.4. Финальное обучение

In [ ]:
itemknn_best_params = select_best_params(itemknn_validation)
itemknn_best_model, itemknn_test_result, _ = fit_best_on_test(
    family="ContentItemKNN",
    best_params=itemknn_best_params,
    factory=make_itemknn,
    train_validation_frame=train_validation,
    test_frame=test,
    fit_kwargs={"items": items},
)
BEST_MODELS["ContentItemKNN"] = itemknn_best_model
METRICS["ContentItemKNN"] = itemknn_test_result
display(pd.DataFrame([itemknn_test_result]))

## 12. Sequential-модель. Transformer (4p)

Предыдущие модели почти не используют порядок событий. Transformer получает
последовательность прошлых товаров и предсказывает следующий. Для истории
$(i_1,\ldots,i_t)$ вход обучающего примера равен $(i_1,\ldots,i_{t-1})$, а целевой
товар — $i_t$.

Соберем модель по шагам:

| Шаг | Что реализуем |
| --- | --- |
| 12.1 | данные: примеры, `Dataset` и `collate_fn` |
| 12.2 | `CausalSelfAttention` — внимание с каузальной маской |
| 12.3 | `FeedForward` — MLP внутри позиции |
| 12.4 | `TransformerBlock` — блок из внимания и MLP |
| 12.5 | `NextItemTransformer` — эмбеддинги, стек блоков и логиты |
| 12.6 | `next_item_loss` — функция потерь |
| 12.7 | `train_epoch` — одна эпоха обучения |
| 12.8 | `TransformerRetriever` — модель отбора кандидатов |

### 12.1. Обучающие примеры, датасет и батчи (1p)

**Что нужно реализовать**

1. `build_sequence_examples` — для каждой позиции истории, начиная со второй, пара
   `(префикс, целевой товар)`. Префикс содержит только товары до целевой позиции и
   обрезается слева до `max_len`; история длины 1 примеров не даёт.
2. `SequenceDataset` — обёртка над списком примеров.
3. `collate_fn` — сборка батча: короткие префиксы дополняются `PAD = 0` **справа** до
   длины самого длинного префикса в батче. Кроме `input_ids` батч содержит:
   * `attention_mask` — `True` на реальных позициях, `False` на PAD;
   * `lengths` — исходные длины префиксов;
   * `targets` — целевые товары.

Дополнять до максимума в батче, а не до `max_len`, заметно дешевле: большинство
историй короткие. `attention_mask` нужна вниманию, чтобы не смотреть на PAD, а
`lengths` — модели, чтобы взять состояние последней реальной позиции.

In [ ]:
def build_sequence_examples(
    histories: Sequence[Sequence[int]],
    max_len: int,
    show_progress: bool = True,
) -> list[tuple[list[int], int]]:
    """Строит пары «префикс истории — следующий товар»."""
    # PUT YOUR CODE HERE
    return


class SequenceDataset(Dataset):
    """Хранит пары «префикс — следующий товар»."""

    examples: list[tuple[list[int], int]]

    def __init__(self, examples: list[tuple[list[int], int]]) -> None:
        self.examples = examples

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int) -> tuple[list[int], int]:
        return self.examples[index]


def collate_fn(batch: Sequence[tuple[list[int], int]]) -> dict[str, Tensor]:
    """Дополняет префиксы PAD = 0 справа и собирает батч."""
    lengths = torch.tensor([len(prefix) for prefix, _ in batch], dtype=torch.long)
    input_ids = torch.zeros((len(batch), int(lengths.max())), dtype=torch.long)
    for row, (prefix, _) in enumerate(batch):
        input_ids[row, : len(prefix)] = torch.tensor(prefix, dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": input_ids.ne(0),
        "lengths": lengths,
        "targets": torch.tensor([target for _, target in batch], dtype=torch.long),
    }


tests.check_sequence_data(build_sequence_examples, SequenceDataset, collate_fn)

### 12.2. Внимание с каузальной маской

Механизм внимания вычисляет

$$Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_h}}+M\right)V.$$

Матрицы $Q$, $K$ и $V$ содержат векторы запросов, ключей и значений; $d_h$ —
размерность одной головы внимания; маска $M$ равна $0$ на разрешённых парах позиций
и `-inf` на запрещённых.

**Что нужно реализовать**

Модуль получает тензор `x` формы `(B, S, D)` и булеву маску `mask` формы `(B, S)`,
где `True` — реальная позиция, `False` — PAD. Нужно:

1. спроецировать вход в $Q$, $K$, $V$ одним линейным слоем и разбить на `n_heads` голов;
2. посчитать $QK^T/\sqrt{d_h}$;
3. поставить `-inf` там, где позиция $t$ смотрела бы в будущее ($j>t$);
4. поставить `-inf` на ключах-заполнителях;
5. взять `softmax`, применить `dropout` и умножить на $V$;
6. склеить головы обратно и применить финальную линейную проекцию.

Это основной механизм модели: каждая позиция собирает информацию из предыдущего
контекста. Каузальная маска гарантирует, что при предсказании товара $i_{t+1}$
модель видит только $i_1,\ldots,i_t$ — иначе она подсмотрела бы ответ. Padding-маска
не даёт заполнителям влиять на реальные позиции.

In [ ]:
class CausalSelfAttention(nn.Module):
    """Многоголовое внимание с каузальной и padding-маской."""

    n_heads: int
    head_dim: int
    qkv: nn.Linear
    projection: nn.Linear
    dropout: nn.Dropout

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model должен делиться на n_heads")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.projection = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor, mask: Tensor) -> Tensor:
        """Возвращает контекстные представления той же формы (B, S, D)."""
        batch_size, seq_len, d_model = x.shape

        # (B, S, 3D) -> три тензора (B, n_heads, S, head_dim)
        query, key, value = ... # PUT YOUR CODE HERE

        weights = ... # PUT YOUR CODE HERE

        # M: позиция t не смотрит в будущее...
        future = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device), diagonal=1
        )
        weights = weights.masked_fill(future, -torch.inf)

        # ...и не смотрит на заполнители.
        weights = weights.masked_fill(~mask[:, None, None, :], -torch.inf)

        weights = self.dropout(torch.softmax(weights, dim=-1))
        context = (weights @ value).transpose(1, 2).reshape(batch_size, seq_len, d_model)
        return self.projection(context)


tests.check_causal_self_attention(CausalSelfAttention)

### 12.3. Position-wise MLP

Внимание смешивает информацию **между** позициями, а MLP выполняет нелинейное
преобразование **внутри** позиции. Вместе они образуют стандартный блок трансформера.

**Что нужно реализовать**

Двухслойный MLP, который применяется к каждой позиции независимо:
`Linear(d -> 4d)` → `GELU` → `Dropout` → `Linear(4d -> d)`. Форма входа и выхода
совпадает: `(B, S, D)`.

In [ ]:
class FeedForward(nn.Module):
    """Position-wise MLP: d -> 4d -> d."""

    layers: nn.Sequential

    def __init__(self, d_model: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.layers = ... # PUT YOUR CODE HERE

    def forward(self, x: Tensor) -> Tensor:
        """Применяет MLP независимо к каждой позиции."""
        return self.layers(x)


tests.check_feed_forward(FeedForward)

### 12.4. Блок трансформера

Трансформер — это последовательность одинаковых блоков. Residual-связь позволяет
градиенту свободно проходить сквозь всю глубину, а Pre-LayerNorm делает обучение
устойчивым. Обратите внимание: если обнулить
обе ветви, блок превращается в тождественное отображение.

**Что нужно реализовать**

Блок в постановке Pre-LayerNorm: нормировка стоит **перед** каждой ветвью, а её
результат прибавляется к входу (residual-связь):

$$x \leftarrow x + Attn(LN(x)), \qquad x \leftarrow x + FF(LN(x)).$$

In [ ]:
class TransformerBlock(nn.Module):
    """Блок трансформера в постановке Pre-LayerNorm."""

    attention_norm: nn.LayerNorm
    attention: CausalSelfAttention
    feed_forward_norm: nn.LayerNorm
    feed_forward: FeedForward
    dropout: nn.Dropout

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.attention_norm = nn.LayerNorm(d_model)
        self.attention = CausalSelfAttention(d_model, n_heads, dropout)
        self.feed_forward_norm = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor, mask: Tensor) -> Tensor:
        """x + Attn(LN(x)), затем x + FF(LN(x))."""
        # PUT YOUR CODE HERE
        return


tests.check_transformer_block(TransformerBlock)

### 12.5. Модель предсказания следующего товара (1p)

Состояние последней позиции — это сжатое представление всей истории пользователя.
Логиты по нему и есть оценки релевантности кандидатов.

**Что нужно реализовать**

1. эмбеддинг товара кодирует его идентификатор, позиционный эмбеддинг — место в
   истории; они складываются. Токен $0$ зарезервирован под PAD, поэтому словарь
   имеет размер `n_items + 1`, а `padding_idx=0` держит вектор заполнителя нулевым;
2. `dropout`, затем стек из `n_layers` блоков, затем финальный `LayerNorm`;
3. из полученных состояний берётся **последняя реальная позиция** каждой
   последовательности (её номер — `lengths - 1`);
4. линейный слой превращает это состояние в логиты по всем товарам.


In [ ]:
class NextItemTransformer(nn.Module):
    """Предсказывает следующий товар по префиксу истории."""

    item_embedding: nn.Embedding
    position_embedding: nn.Embedding
    dropout: nn.Dropout
    blocks: nn.ModuleList
    norm: nn.LayerNorm
    output: nn.Linear

    def __init__(
        self,
        n_items: int,
        max_len: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        # Токен 0 зарезервирован под PAD, поэтому словарь на единицу длиннее каталога.
        self.item_embedding = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            TransformerBlock(d_model, n_heads, dropout) for _ in range(n_layers)
        )
        self.norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, n_items)

    def forward(
        self,
        input_ids: Tensor,
        attention_mask: Tensor,
        lengths: Tensor,
    ) -> Tensor:
        """Логиты следующего товара по состоянию последней реальной позиции."""
        # PUT YOUR CODE HERE
        return


tests.check_next_item_transformer(NextItemTransformer)

### 12.6. Функция потерь

Задача «предсказать следующий товар» — это классификация на $|I|$ классов, поэтому
берём кросс-энтропию:

$$L=-\frac1B\sum_{b=1}^{B}\log
\frac{\exp z_{b,y_b}}{\sum_{j\in I}\exp z_{b,j}}.$$

Здесь $B$ — размер батча, $y_b$ — истинный следующий товар в примере $b$,
$z_{b,j}$ — логит товара $j$, а $I$ — множество товаров-кандидатов.

Функция потерь штрафует модель, если логит истинного товара мал относительно логитов
остальных товаров.

**Что нужно реализовать**

Функцию `next_item_loss(logits, targets)`, возвращающую скалярный тензор.

In [ ]:
def next_item_loss(logits: Tensor, targets: Tensor) -> Tensor:
    """Перекрёстная энтропия по реальным товарам.

    targets используют токены 1..n_items, тогда как классы logits
    индексируются от 0 до n_items-1.
    """
    # PUT YOUR CODE HERE
    return


tests.check_next_item_loss(next_item_loss)

### 12.7. Одна эпоха обучения (1p)

**Что нужно реализовать**

Проход по всем батчам: перенести батч на устройство, посчитать логиты и лосс,
выполнить `zero_grad → backward → step`, вернуть средний лосс за эпоху.

Батч переносится на устройство непосредственно перед прямым проходом — весь датасет
на GPU заранее не копируется. Средний лосс возвращается, чтобы было видно, что
обучение идёт: `tqdm` показывает его по ходу эпохи.

In [ ]:
def train_epoch(
    model: nn.Module,
    dataloader: Iterable[Mapping[str, Tensor]],
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    desc: str,
    show_progress: bool = True,
) -> float:
    """Обучает модель один проход по данным и возвращает средний лосс."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    progress = tqdm(dataloader, desc=desc, disable=not show_progress)
    for batch in progress:

        # PUT YOUR CODE HERE

        total_loss += loss.item()
        n_batches += 1
        progress.set_postfix(loss=f"{total_loss / n_batches:.4f}")
    return total_loss / max(n_batches, 1)


tests.check_train_epoch(train_epoch)

### 12.8. `TransformerRetriever` (1p)

Соберём всё вместе в модель отбора кандидатов:

1. товары нумеруются с единицы, потому что $0$ занят под PAD. Удобно взять
   `item_to_token[item] = item_to_index[item] + 1`;
2. история пользователя упорядочивается по `timestamp`;
3. обучение: примеры → `SequenceDataset` → `DataLoader(collate_fn=collate_fn)` →
   `train_epoch` нужное число эпох;
4. выдача: пользователи обрабатываются батчами, история получает `-inf`,
   дальше работает уже знакомый `_rank_by_scores`. Пользователи без истории
   получают популярные товары.


In [ ]:
class TransformerRetriever(BaseRecommender):
    """Последовательная модель: предсказывает следующий товар по истории."""

    max_len: int
    d_model: int
    n_heads: int
    n_layers: int
    dropout: float
    epochs: int
    learning_rate: float
    device: torch.device
    seed: int
    item_to_token: dict[str, int]
    histories: dict[str, list[int]]
    model: NextItemTransformer

    def __init__(
        self,
        max_len: int = 30,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        dropout: float = 0.1,
        epochs: int = 1,
        batch_size: int = 512,
        learning_rate: float = 1e-3,
        device: str | None = None,
        seed: int = 42,
        show_progress: bool = True,
    ) -> None:
        super().__init__("Transformer", show_progress)
        self.max_len = max_len
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.seed = seed

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        """Строит истории пользователей и обучает модель."""
        torch.manual_seed(self.seed)
        self.item_to_token = {
            item: index + 1 for item, index in self.item_to_index.items()
        }
        ordered = interactions.sort(["user_id", "timestamp"], maintain_order=True)
        grouped = ordered.group_by("user_id", maintain_order=True).agg(
            pl.col("item_id")
        )
        self.histories = {
            user: [self.item_to_token[item] for item in item_list]
            for user, item_list in zip(
                grouped["user_id"].to_list(), grouped["item_id"].to_list()
            )
        }

        self.model = ... # PUT YOUR CODE HERE
        examples = ... # PUT YOUR CODE HERE

        if not examples or self.epochs <= 0:
            return

        dataloader = ... # PUT YOUR CODE HERE

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate)
        for epoch in range(self.epochs):
            train_epoch(
                self.model,
                dataloader,
                optimizer,
                self.device,
                f"Transformer: эпоха {epoch + 1}/{self.epochs}",
                self.show_progress,
            )

    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        """Батчами считает логиты; пользователи без истории получают популярные товары."""
        self.model.eval()
        return self._recommend_in_batches(
            user_ids, topn, lambda user: bool(self.histories.get(user)), self._rank_known
        )

    def _rank_known(self, user_ids: list[str], topn: int) -> list[list[str]]:
        """Считает логиты для пользователей с историей и отбирает top-K."""
        if not user_ids:
            return []
        batch = collate_fn(
            [(self.histories[user][-self.max_len :], 0) for user in user_ids]
        )
        with torch.no_grad():
            logits = self.model(
                batch["input_ids"].to(self.device),
                batch["attention_mask"].to(self.device),
                batch["lengths"].to(self.device),
            )
        scores = logits.float().cpu().numpy()
        for row, user in enumerate(user_ids):
            scores[row, self._seen_indices(user)] = -np.inf
        return self._rank_by_scores(scores, self.items, topn)


tests.check_transformer_retriever(TransformerRetriever)

### 12.9. Подбор гиперпараметров


In [ ]:
transformer_grid: list[dict[str, Any]] = [
    {
        "d_model": d_model,
        "n_layers": n_layers,
        "n_heads": 4,
        "max_len": 50,
        "epochs": 5,
        "batch_size": 256,
    }
    for d_model in [32, 64]
    for n_layers in [2, 4]
]


def make_transformer(params: Mapping[str, Any]) -> BaseRecommender:
    return TransformerRetriever(
        d_model=int(params["d_model"]),
        n_layers=int(params["n_layers"]),
        n_heads=int(params["n_heads"]),
        max_len=int(params["max_len"]),
        epochs=int(params["epochs"]),
        batch_size=int(params["batch_size"]),
    )


transformer_validation = validation_search(
    family="Transformer",
    parameter_grid=transformer_grid,
    factory=make_transformer,
    train_frame=train,
    validation_frame=validation,
)
VALIDATION_RESULTS["Transformer"] = transformer_validation
display(transformer_validation)

### 12.10. Финальное обучение


In [ ]:
transformer_best_params = select_best_params(transformer_validation)
transformer_best_model, transformer_test_result, _ = fit_best_on_test(
    family="Transformer",
    best_params=transformer_best_params,
    factory=make_transformer,
    train_validation_frame=train_validation,
    test_frame=test,
)
BEST_MODELS["Transformer"] = transformer_best_model
METRICS["Transformer"] = transformer_test_result
display(pd.DataFrame([transformer_test_result]))

## 13. Итоговое сравнение на `test` (1p)

Каждая запись словаря `METRICS` уже получена в соответствующем разделе после выбора
параметров по `Recall@20` на `validation` и обучения лучшей конфигурации на
`train + validation`. Здесь мы только собираем накопленные записи в общую
таблицу: ни повторного обучения, ни повторного обращения к `test` не происходит.

In [ ]:
test_results = (
    pd.DataFrame(METRICS.values())
    .sort_values(["recall@20", "ndcg@20"], ascending=False, kind="stable")
    .reset_index(drop=True)
)
display(test_results.style.background_gradient(cmap="Blues"))

Опишите результаты и выводы, к которым вы пришли в процессе работы.